# 🏪 Model Prediksi Kebutuhan Stok (Decision Tree) - Toko Setia Ciawi
Notebook ini berisi dokumentasi dan implementasi proses analisis data dan pemodelan regresi untuk memprediksi penjualan mingguan produk menggunakan algoritma **Decision Tree Regressor**.

### Alur Analisis:
1. **Load Data**: Membaca dataset historis transaksi `data.csv`.
2. **Pembersihan Data (Data Cleaning)**: Mengubah desimal koma menjadi titik dan menyamakan nama barang yang typo.
3. **Preprocessing & Feature Engineering**: Agregasi penjualan mingguan per produk dan pembuatan lag feature (`penjualan_bulan_lalu`).
4. **Pembangunan Pipeline ML**: Menggabungkan `StandardScaler` dan `DecisionTreeRegressor` (max depth = 4).
5. **Evaluasi Model**: 5-Fold Cross Validation & perhitungan Mean Absolute Error (MAE).
6. **Penyimpanan Pipeline**: Menyimpan objek pipeline utuh ke file `model.joblib` menggunakan library `joblib`.

In [ ]:
# 1. Import Library yang Dibutuhkan
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import joblib

## 1. Load Data & Pembersihan Data
Membaca file dataset `data.csv` yang berisi riwayat penjualan toko kelontong.

In [ ]:
# Membaca dataset
df = pd.read_csv('data.csv')

print(f"Dataset berhasil dimuat. Total data: {df.shape[0]} baris, {df.shape[1]} kolom.")
df.head()

In [ ]:
# === PROSES DATA CLEANING ===
# 1. Bersihkan pemisah desimal koma menjadi titik pada kolom Jumlah
df['Jumlah'] = df['Jumlah'].astype(str).str.replace(',', '.').astype(float)

# 2. Standarisasi nama produk yang typo / duplikat
mapping_typo = {
    'Rokok Jarum Cokelat': 'Rokok Jarum Coklat',
    'Rokok Jarim Coklat': 'Rokok Jarum Coklat',
    'Jarum Super': 'Rokok Jarum Super',
    'Rokok Super': 'Rokok Jarum Super',
    'Rokok GGM': 'Rokok Garam Merah',
    'Garam': 'Rokok Garam Merah',
    'Gula Pasir Pasir': 'Gula Pasir',
}
df['Nama_barang'] = df['Nama_barang'].replace(mapping_typo)

df.loc[df['Satuan'] == 'Batang', 'Nama_barang'] = df.loc[df['Satuan'] == 'Batang', 'Nama_barang'] + ' (batang)'

print(f"Dataset berhasil dibersihkan. Total barang unik sekarang: {df['Nama_barang'].nunique()}")
df.head()

## 2. Preprocessing & Feature Engineering
Mengubah data transaksi mentah menjadi fitur input untuk model prediksi stok.

In [ ]:
df_prep = df.copy()

df_prep['Jumlah'] = pd.to_numeric(df_prep['Jumlah'], errors='coerce').fillna(0.0)

# Ekstraksi fitur tanggal
df_prep['tanggal'] = pd.to_datetime(df_prep['Tanggal'])
df_prep['tahun'] = df_prep['tanggal'].dt.year
df_prep['bulan'] = df_prep['tanggal'].dt.month
df_prep['hari'] = df_prep['tanggal'].dt.day

# Menentukan minggu ke berapa dalam bulan (1 s/d 5)
df_prep['minggu_ke'] = ((df_prep['hari'] - 1) // 7) + 1
df_prep['minggu_ke'] = df_prep['minggu_ke'].clip(1, 5)

# Encoding nama barang menjadi ID numerik
unique_items = df_prep['Nama_barang'].unique()
item_to_id = {name: idx + 1 for idx, name in enumerate(unique_items)}
df_prep['barang_id'] = df_prep['Nama_barang'].map(item_to_id)

# Agregasi penjualan mingguan per produk
dataset = df_prep.groupby(['tahun', 'bulan', 'minggu_ke', 'barang_id']).agg(
    jumlah_terjual=('Jumlah', 'sum')
).reset_index()

# Tambahkan Fitur Lag: penjualan_bulan_lalu
monthly_sales = df_prep.groupby(['tahun', 'bulan', 'barang_id'])['Jumlah'].sum().reset_index()
monthly_sales.rename(columns={'Jumlah': 'penjualan_bulan_lalu'}, inplace=True)

dataset['tahun_lalu'] = dataset['tahun']
dataset['bulan_lalu'] = dataset['bulan'] - 1

jan_mask = dataset['bulan'] == 1
dataset.loc[jan_mask, 'tahun_lalu'] = dataset['tahun'] - 1
dataset.loc[jan_mask, 'bulan_lalu'] = 12

dataset = pd.merge(
    dataset,
    monthly_sales,
    left_on=['tahun_lalu', 'bulan_lalu', 'barang_id'],
    right_on=['tahun', 'bulan', 'barang_id'],
    how='left',
    suffixes=('', '_temp')
)

dataset.drop(columns=['tahun_temp', 'bulan_temp', 'tahun_lalu', 'bulan_lalu'], errors='ignore', inplace=True)
dataset['penjualan_bulan_lalu'] = dataset['penjualan_bulan_lalu'].fillna(0)
dataset = dataset.sort_values(by=['tahun', 'bulan', 'minggu_ke', 'barang_id']).reset_index(drop=True)

features = ['barang_id', 'bulan', 'minggu_ke', 'penjualan_bulan_lalu']

print(f"Data berhasil dipreproses. Jumlah dataset teragregasi: {dataset.shape[0]} baris.")
print(f"Fitur input yang digunakan: {features}")
dataset.head()

## 3. Pembangunan Pipeline & Evaluasi Model
Membangun pipeline model ML dan mengevaluasi performa menggunakan 5-Fold Cross Validation.

In [ ]:
X = dataset[features].values
y = dataset['jumlah_terjual'].values

kf = KFold(n_splits=5, shuffle=True, random_state=42)
mae_scores = []

print("=== Memulai 5-Fold Cross Validation ===")
for fold, (train_idx, test_idx) in enumerate(kf.split(X), 1):
    X_tr, X_te = X[train_idx], X[test_idx]
    y_tr, y_te = y[train_idx], y[test_idx]
    
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', DecisionTreeRegressor(max_depth=4, random_state=42))
    ])
    
    pipeline.fit(X_tr, y_tr)
    
    preds = pipeline.predict(X_te)
    mae = mean_absolute_error(y_te, preds)
    mae_scores.append(mae)
    print(f"Fold {fold} - MAE: {mae:.4f} unit laku")

mean_mae = np.mean(mae_scores)
print(f"\nRata-rata MAE Keseluruhan: {mean_mae:.4f} unit laku")

## 4. Final Model Training & Visualisasi
Melatih pipeline model pada seluruh data yang tersedia dan melakukan visualisasi pohon keputusan (*decision tree*).

In [ ]:
final_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', DecisionTreeRegressor(max_depth=4, random_state=42))
])
final_pipeline.fit(X, y)

print("Model Pipeline akhir berhasil dilatih pada seluruh data.")

regressor_model = final_pipeline.named_steps['regressor']
plt.figure(figsize=(20, 10))
plot_tree(
    regressor_model, 
    feature_names=features, 
    filled=True, 
    rounded=True, 
    fontsize=10
)
plt.title("Visualisasi Struktur Pohon Keputusan (Decision Tree)", fontsize=16)
plt.show()

## 5. Simpan Pipeline ke File `.joblib`
Menyimpan objek pipeline terlatih beserta daftar fitur dan skor evaluasi menggunakan format `.joblib` untuk dideploy di Streamlit.

In [ ]:
model_save_path = 'model.joblib'

model_data = {
    "pipeline": final_pipeline,
    "features": features,
    "mean_mae": mean_mae,
    "data_count": len(dataset)
}

joblib.dump(model_data, model_save_path)
print(f"Sukses! Model Pipeline berhasil disimpan di: {model_save_path}")